# Segmentation Analysis

In this section, we investigate whether the treatment effect varies
across different user segments.

Understanding heterogeneous treatment effects can provide insights into:
- which users benefit most from the treatment
- potential targeting opportunities
- differences in user behavior across segments

In [ ]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))

df = pd.read_csv("../data/processed/ab_data_cleaned.csv")

df.head()

---
## Segment-Level Conversion Analysis

We evaluate conversion performance separately across different user
segments to assess whether the treatment effect is consistent.

### Device Type

In [ ]:
device_summary = (
    df.groupby(["device_type", "group"])["converted"]
    .mean()
    .unstack()
)

device_summary["absolute_uplift"] = (
    device_summary["treatment"]
    - device_summary["control"]
)

device_summary["relative_uplift"] = (
    device_summary["absolute_uplift"]
    / device_summary["control"]
)

device_summary

In [ ]:
from src import plot_segment_conversion

plot_segment_conversion(df, "device_type")

### Gender

In [ ]:
gender_summary = (
    df.groupby(["gender", "group"])["converted"]
    .mean()
    .unstack()
)

gender_summary["absolute_uplift"] = (
    gender_summary["treatment"]
    - gender_summary["control"]
)

gender_summary["relative_uplift"] = (
    gender_summary["absolute_uplift"]
    / gender_summary["control"]
)

gender_summary

In [ ]:
plot_segment_conversion(df, "gender")

### Location

In [ ]:
location_summary = (
    df.groupby(["location", "group"])["converted"]
    .mean()
    .unstack()
)

location_summary["absolute_uplift"] = (
    location_summary["treatment"]
    - location_summary["control"]
)

location_summary["relative_uplift"] = (
    location_summary["absolute_uplift"]
    / location_summary["control"]
)

location_summary

In [ ]:
plot_segment_conversion(df, "location")

Overall, the treatment effect appears broadly consistent across all evaluated segments, although moderate variation in uplift magnitude is observed.

---
## Statistical Testing by Segment

We now perform separate one-sided two-proportion z-tests within each
segment to evaluate whether the treatment group achieves a higher
conversion rate than the control group.

For each segment, the hypotheses are:

$H_0$: $p_{treatment} = p_{control}$

$H_1$: $p_{treatment} > p_{control}$

where the proportions are computed using only observations belonging
to the corresponding subgroup.

Because multiple hypothesis tests are performed, some significant results
may occur by chance alone.

Therefore, results should be interpreted cautiously and evaluated
collectively rather than individually.

In [ ]:
from src import segment_ztest

device_tests = segment_ztest(df, "device_type")
device_tests

In [ ]:
gender_tests = segment_ztest(df, "gender")
gender_tests

In [ ]:
location_tests = segment_ztest(df, "location")
location_tests

All segment-level tests produced extremely small p-values, providing strong evidence against the null hypothesis across all evaluated subgroups.

Additionally, the estimated uplifts remain relatively consistent between segments, suggesting that the treatment effect is broadly robust rather than concentrated within a specific subgroup.

Although uplift magnitudes vary slightly across segments, the observed
differences remain relatively moderate across user groups.

---
## Interaction Effect Analysis

While subgroup analysis evaluates treatment effects separately within
each segment, interaction modeling allows us to formally test whether
the treatment effect differs across user groups.

To illustrate this approach, we perform an interaction analysis using
device type as a representative segmentation variable.

The same methodology could be extended to other categorical features
such as gender or location.

We fit a logistic regression model including:
- treatment assignment
- device type
- interaction terms between treatment and device type

The interaction coefficients measure whether the treatment effect
changes relative to the baseline device category.

A statistically significant interaction term would indicate evidence of
heterogeneous treatment effects across devices.

In [ ]:
from src import logistic_segment_interaction

interaction_results = logistic_segment_interaction(df, "device_type")

interaction_results

The interaction terms between treatment assignment and device type are
small and statistically non-significant.

These results provide no evidence that the treatment effect differs
meaningfully across device categories.

This suggests that the observed uplift is reasonably consistent across
desktop, mobile, and tablet users.

---
## Conclusion

The segmentation analysis suggests that the treatment effect is broadly
consistent across the evaluated user groups.

Although uplift magnitudes vary moderately between segments, the treatment
improves conversion performance across all analyzed subpopulations.

All segment-level tests produced extremely small p-values, which is
expected given the large sample size and the consistently positive
treatment effect.

The interaction terms between treatment assignment and
device type were small and statistically non-significant,
providing no evidence that the treatment effect differs meaningfully
across device categories.

Overall, the results suggest that the treatment effect is reasonably
stable across the evaluated segments.